In [5]:
import requests
from datetime import datetime
import os
from dotenv import load_dotenv
import pandas as pd
import json
import re
import html
load_dotenv()
api_key = os.getenv("API_KEY")

In [4]:
class ExtractChampionData:

    def __init__(self):
        self.last_version = self.get_latest_version()
        self.list_champ = self.get_all_champ_general_data()
        self.list_champ = list(self.list_champ.keys())

    def get_latest_version(self):
        """Get the last version of the game"""
        url = "https://ddragon.leagueoflegends.com/api/versions.json"
        rep = requests.get(url).json()
        latest = rep[0]
        return latest

    def get_all_champ_general_data(self):
        """Get the champion data"""
        url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/data/fr_FR/champion.json"
        resp = requests.get(url).json()
        return resp['data']
    
    def get_details_champ_data(self):
        """Get detail champion data"""
        data = []
        for champ in self.list_champ:
            url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/data/fr_FR/champion/{champ}.json"
            resp = requests.get(url).json()['data']
            data.append(resp[champ])
        return data
    
    def get_champion_mastery(self, puuid : str):
        """Get all champion mastery entries sorted by number of champion points descending."""

        api_url = f"https://euw1.api.riotgames.com/lol/champion-mastery/v4/champion-masteries/by-puuid/{puuid}?api_key={api_key}"
        resp = requests.get(api_url).json()
        return resp
        

    # def download_all_png_champion(self):
    #     """Download all champions pictures"""
    #     for champ in self.list_champ:
    #         url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/img/champion/{champ}.png"
    #         path_save = fr"C:\Users\najim\Documents\Projets\LeagueOfLegends\images\{champ}.png"

    #         resp = requests.get(url)
    #         with open(path_save, "wb") as file:
    #             file.write(resp.content)

In [89]:
class TransformChampionData:

    def __init__(self):
        self.keys_to_drop = ['image', 'skins', 'blurb']
        self.table_champion = ['key', 'name', 'title', 'lore', 'tags', 'partype']
        self.table_champ_passive = ['key', 'passive']
        self.table_champ_info = ['key', 'info', 'allytips', 'enemytips']
        self.table_champ_spells = ['key', 'spells']
        self.table_champ_stats = ['key', 'stats']
    
    def drop_keys(self, data : list) -> list:
        new_data = []
        for dict_champ in data:
            dict_champ = {k: v for k,v in dict_champ.items() if k not in self.keys_to_drop}
            new_data.append(dict_champ)
        return new_data
    
    def dispatch_data(self, data : list):
        table_champion = []
        table_champ_passive = []
        table_champ_info = []
        table_champ_spells = []
        table_champ_stats = []
        
        for dict_champ in data:
            champion = {k: v for k,v in dict_champ.items() if k in self.table_champion}
            champ_passive = {k: v for k,v in dict_champ.items() if k in self.table_champ_passive}
            champ_info = {k: v for k,v in dict_champ.items() if k in self.table_champ_info}
            champ_spells = {k: v for k,v in dict_champ.items() if k in self.table_champ_spells}
            champ_stats = {k: v for k,v in dict_champ.items() if k in self.table_champ_stats}

            table_champion.append(champion)
            table_champ_passive.append(champ_passive)
            table_champ_info.append(champ_info)
            table_champ_spells.append(champ_spells)
            table_champ_stats.append(champ_stats)
            
        return table_champion, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats
    
    def split_stats(self, data : list):
        stats_cst, stats_up = [], []
        for stat in data:
            cst = {k:v for k,v in stat.items() if "perlevel" not in k}
            up = {k: v for k, v in stat.items() if "perlevel" in k or "key" in k}
            stats_cst.append(cst)
            stats_up.append(up)
        return stats_cst, stats_up

    def listing_into_text(self, data : list, key : str, sep : str = " "):
        new_data = []
        for champ in data:
            new_value = f"{sep}".join(champ[key])
            champ[key] = self._clean_text(new_value)
            new_data.append(champ)
        return new_data

    def dict_into_first_level(self, data : list, key_to_flat : str) -> list:
        new_data = []
        for champ in data:
            base = {k: v for k, v in champ.items() if k != key_to_flat}
            base.update(champ.get(key_to_flat, {}))
            new_data.append(base)
        return new_data
    
    def pop_key(self, data : list, key_to_remove : list[str]) -> list:
        new_data = []
        for i in data:
            for key in key_to_remove:
                d = i.pop(key, None)
            new_data.append(i)
        return new_data
    
    def _clean_text(self, desc : str) -> str:
        desc = re.sub(r"<[^>]+>|<[a-z]>", " ", desc)
        desc = re.sub(r"\xa0", " ", desc)
        desc = re.sub(r"_ClientTooltipWrapper|Wrapper", "", desc)
        desc = re.sub(r"\{\{.*?\}\}", "XX", desc)
        desc = re.sub(r"XX$", "", desc)
        desc = re.sub(r"[ ]+", " ", desc)
        return desc

    def text_cleaning(self, data : list, key : str) -> list:
        clean = []
        for champ in data:
            desc = self._clean_text(champ[key])
            champ[key] = desc
            clean.append(champ)
        return clean
    
    def clean_spells_data(self, data: list):
        new_data = []
        for champ in data:
            idchamp = champ['key']
            spells = champ['spells']

            spells = self.pop_key(
                spells,
                key_to_remove=['leveltip', 'description', 'cooldown', 'cost', 'datavalues',
                    'effect', 'vars', 'image', 'effectBurn', 'range'])

            spells_clean = self.text_cleaning(spells, 'tooltip')
            spells_clean = self.text_cleaning(spells, 'name')
            spells_clean = self.text_cleaning(spells, 'id')

            for spell in spells_clean:
                spell['key'] = idchamp
                new_data.append(spell)

        return new_data
    
    def transform_to_df(self, data : list) -> pd.DataFrame:
        """Transforme une liste de dictionnaire en dataframe pandas"""
        return pd.DataFrame(data)
        

In [90]:
def pipeline_champion():
    """Pipeline d'application des méthodes d'extraction, de transformation et de chargement"""
    
    extract = ExtractChampionData()
    transform = TransformChampionData()

    details = extract.get_details_champ_data()
    details = transform.drop_keys(details)
    
    table_champion, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats = transform.dispatch_data(details)
    
    table_champion = transform.listing_into_text(table_champion, "tags", sep=' / ')
    table_champion = transform.text_cleaning(table_champion, key='lore')
    table_champion = transform.text_cleaning(table_champion, key='title')
    
    table_champ_info = transform.listing_into_text(table_champ_info, key="enemytips", sep=' ')
    table_champ_info = transform.listing_into_text(table_champ_info, key="allytips", sep=' ')
    table_champ_info = transform.dict_into_first_level(table_champ_info, key_to_flat='info')

    table_champ_passive = transform.dict_into_first_level(table_champ_passive, key_to_flat='passive')
    table_champ_passive = transform.pop_key(table_champ_passive, key_to_remove=['image'])
    table_champ_passive = transform.text_cleaning(table_champ_passive, key='name')
    table_champ_passive = transform.text_cleaning(table_champ_passive, key='description')

    table_champ_stats = transform.dict_into_first_level(table_champ_stats, key_to_flat='stats')
    table_champ_stats, table_champ_stats_up = transform.split_stats(table_champ_stats)

    table_champ_spells = transform.pop_key(table_champ_spells, key_to_remove= ['leveltip', 'description', 'cooldown', 'cost', 'datavalues', 
                                                                               'effect', 'vars', 'image', 'effectBurn', 'range'])
    table_champ_spells = transform.clean_spells_data(table_champ_spells)

    table_champion = transform.transform_to_df(table_champion)
    table_champ_info = transform.transform_to_df(table_champ_info)
    table_champ_passive = transform.transform_to_df(table_champ_passive)
    table_champ_stats = transform.transform_to_df(table_champ_stats)
    table_champ_stats_up = transform.transform_to_df(table_champ_stats_up)
    table_champ_spells = transform.transform_to_df(table_champ_spells)
    
    return table_champion, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats, table_champ_stats_up

table_champion, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats, table_champ_stats_up = pipeline_champion()

In [91]:
table_champ_spells

,id,name,tooltip,maxrank,cooldownBurn,costBurn,costType,maxammo,rangeBurn,resource,key
0,AatroxQ,Épée des Darkin,"Aatrox abat son épée, infligeant XX pts de dég...",5,14/12/10/8/6,0,Pas de coût,-1,25000,Pas de coût,266
1,AatroxW,Chaînes infernales,"Aatrox lance une chaîne, ralentissant le premi...",5,20/18/16/14/12,0,Pas de coût,-1,825,Pas de coût,266
2,AatroxE,Ruée obscure,Passive : Aatrox récupère des PV équivalents ...,5,9/8/7/6/5,0,Pas de coût,-1,25000,Pas de coût,266
3,AatroxR,Fossoyeur des mondes,"Aatrox révèle sa vraie forme démoniaque, effra...",3,120/100/80,0,Pas de coût,-1,25000,Pas de coût,266
4,AhriQ,Orbe d'illusion,"Ahri lance son orbe et le ramène vers elle, in...",5,7,55/65/75/85/95,{{ abilityresourcename }},-1,970,{{ cost }} {{ abilityresourcename }},103
...,...,...,...,...,...,...,...,...,...,...,...
683,ZoeR,Bond dimensionnel,Zoé se téléporte à un endroit proche pendant 1...,3,11/8/5,40,{{ abilityresourcename }},-1,575,{{ cost }} {{ abilityresourcename }},142
684,ZyraQ,Épines funestes,Zyra répand des ronces dans le sol et provoque...,5,7/6.5/6/5.5/5,55,{{ abilityresourcename }},-1,800,{{ cost }} {{ abilityresourcename }},143
685,ZyraW,Croissance incontrôlée,Zyra plante une graine qui dure XX sec. Quand ...,5,0,0,1 graine,2,850,1 graine,143
686,ZyraE,Racines fixatrices,Zyra lance des racines qui immobilisent les en...,5,11,70/75/80/85/90,{{ abilityresourcename }},-1,1100,{{ cost }} {{ abilityresourcename }},143


In [92]:
for i in table_champ_spells['tooltip']:
    print(i)

Aatrox abat son épée, infligeant XX pts de dégâts physiques . Si l'ennemi est touché par le tranchant, il est brièvement projeté dans les airs et subit XX pts de dégâts à la place. Cette compétence peut être réactivée deux fois, chaque coup changeant de forme et infligeant 25% de dégâts de plus que la précédente.
Aatrox lance une chaîne, ralentissant le premier ennemi touché de XX% pendant XX sec et infligeant XX pts de dégâts physiques . Les champions et les grands monstres doivent quitter la zone d'effet dans les XX sec sous peine d'être ramenés de force au point d'impact et de subir à nouveau les dégâts.
 Passive : Aatrox récupère des PV équivalents à XX des dégâts qu'il inflige aux champions. Active : Aatrox se rue dans une direction. Il peut utiliser cette compétence tout en lançant ses autres compétences.
Aatrox révèle sa vraie forme démoniaque, effrayant les sbires proches pendant XX sec et gagnant +XX% de vitesse de déplacement (ce bonus diminue en XX sec). Il gagne aussi +XX% 

In [93]:
table_champion

,key,name,title,lore,tags,partype
0,266,Aatrox,Épée des Darkin,"Autrefois, Aatrox et ses frères étaient honoré...",Fighter,Puits de sang
1,103,Ahri,Renarde à neuf queues,"Connectée à la magie du royaume spirituel, Ahr...",Mage / Assassin,Mana
2,84,Akali,Assassine rebelle,Ayant abandonné l'Ordre Kinkou et le titre de ...,Assassin,Énergie
3,166,Akshan,Sentinelle rebelle,"Se jouant du danger, Akshan combat le mal sans...",Marksman / Assassin,Mana
4,12,Alistar,Minotaure,Alistar est un guerrier redoutable cherchant à...,Tank / Support,Mana
...,...,...,...,...,...,...
167,221,Zeri,Étincelle de Zaun,Zeri est une jeune femme vive et téméraire ori...,Marksman,Mana
168,115,Ziggs,Expert des Hexplosifs,Amoureux des grosses bombes et des mèches cour...,Mage,Mana
169,26,Zilean,Gardien du Temps,"Naguère puissant mage d'Icathia, Zilean devint...",Support / Mage,Mana
170,142,Zoé,Manifestation du Crépuscule,"Incarnation de l'espièglerie, de l'imagination...",Mage,Mana


In [94]:
df_merge = pd.merge(table_champ_spells[['key', 'id']], table_champion[['key', 'name']],
                    on='key')
df_merge

,key,id,name
0,266,AatroxQ,Aatrox
1,266,AatroxW,Aatrox
2,266,AatroxE,Aatrox
3,266,AatroxR,Aatrox
4,103,AhriQ,Ahri
...,...,...,...
683,142,ZoeR,Zoé
684,143,ZyraQ,Zyra
685,143,ZyraW,Zyra
686,143,ZyraE,Zyra


In [103]:
candidate = None
count = 0

for idx,i,j in zip(df_merge.index, df_merge['id'], df_merge['name']):

    if count == 0:
        candidate = j
    
    if j == candidate:
        count += 1
    else:
        candidate = j
        count = 1
    
    if count == 1:
        df_merge.at[idx, 'new_name_spell'] = j + "Q"
    
    elif count == 2:
        df_merge.at[idx, 'new_name_spell'] = j + "W"
    
    elif count == 3:
        df_merge.at[idx, 'new_name_spell'] = j + "E"
    
    elif count == 4:
        df_merge.at[idx, 'new_name_spell'] = j + "R"

In [104]:
df_merge

,key,id,name,new_name_spell
0,266,AatroxQ,Aatrox,AatroxQ
1,266,AatroxW,Aatrox,AatroxW
2,266,AatroxE,Aatrox,AatroxE
3,266,AatroxR,Aatrox,AatroxR
4,103,AhriQ,Ahri,AhriQ
...,...,...,...,...
683,142,ZoeR,Zoé,ZoéR
684,143,ZyraQ,Zyra,ZyraQ
685,143,ZyraW,Zyra,ZyraW
686,143,ZyraE,Zyra,ZyraE


In [105]:
for i,j,u in zip(df_merge['id'], df_merge['name'], df_merge['new_name_spell']):
    print(f"{i} / {j} / {u}")

AatroxQ / Aatrox / AatroxQ
AatroxW / Aatrox / AatroxW
AatroxE / Aatrox / AatroxE
AatroxR / Aatrox / AatroxR
AhriQ / Ahri / AhriQ
AhriW / Ahri / AhriW
AhriE / Ahri / AhriE
AhriR / Ahri / AhriR
AkaliQ / Akali / AkaliQ
AkaliW / Akali / AkaliW
AkaliE / Akali / AkaliE
AkaliR / Akali / AkaliR
AkshanQ / Akshan / AkshanQ
AkshanW / Akshan / AkshanW
AkshanE / Akshan / AkshanE
AkshanR / Akshan / AkshanR
Pulverize / Alistar / AlistarQ
Headbutt / Alistar / AlistarW
AlistarE / Alistar / AlistarE
FerociousHowl / Alistar / AlistarR
AmbessaQ / Ambessa / AmbessaQ
AmbessaW / Ambessa / AmbessaW
AmbessaE / Ambessa / AmbessaE
AmbessaR / Ambessa / AmbessaR
BandageToss / Amumu / AmumuQ
AuraofDespair / Amumu / AmumuW
Tantrum / Amumu / AmumuE
CurseoftheSadMummy / Amumu / AmumuR
FlashFrost / Anivia / AniviaQ
Crystallize / Anivia / AniviaW
Frostbite / Anivia / AniviaE
GlacialStorm / Anivia / AniviaR
AnnieQ / Annie / AnnieQ
AnnieW / Annie / AnnieW
AnnieE / Annie / AnnieE
AnnieR / Annie / AnnieR
ApheliosQ / Aphelio

In [109]:
df_merge_test = pd.merge(table_champ_spells[['key', 'id']], table_champion[['key', 'name']],
                    on='key')
df_merge_test

,key,id,name
0,266,AatroxQ,Aatrox
1,266,AatroxW,Aatrox
2,266,AatroxE,Aatrox
3,266,AatroxR,Aatrox
4,103,AhriQ,Ahri
...,...,...,...
683,142,ZoeR,Zoé
684,143,ZyraQ,Zyra
685,143,ZyraW,Zyra
686,143,ZyraE,Zyra


In [112]:
letters = ['Q', 'W', 'E', 'R']

# numérotation 0,1,2,3 par groupe de 'name'
df_merge_test['spell_idx'] = df_merge_test.groupby('name').cumcount()
# display(df_merge_test)
# on mappe 0→Q, 1→W, 2→E, 3→R
df_merge_test['new_name_spell'] = df_merge_test['name'] + df_merge_test['spell_idx'].map(lambda x: letters[x])
df_merge_test

,key,id,name,spell_idx,new_name_spell
0,266,AatroxQ,Aatrox,0,AatroxQ
1,266,AatroxW,Aatrox,1,AatroxW
2,266,AatroxE,Aatrox,2,AatroxE
3,266,AatroxR,Aatrox,3,AatroxR
4,103,AhriQ,Ahri,0,AhriQ
...,...,...,...,...,...
683,142,ZoeR,Zoé,3,ZoéR
684,143,ZyraQ,Zyra,0,ZyraQ
685,143,ZyraW,Zyra,1,ZyraW
686,143,ZyraE,Zyra,2,ZyraE
